# Silver Layer - Rental Processing

This notebook processes CDC events from the Bronze layer and merges them into the Silver rental table with schema evolution support and **schema drift detection**.

In [ ]:
%run ../helpers/NB_catalog_helpers

In [ ]:
%run ../helpers/NB_schema_drift_helpers

In [ ]:
%run ../helpers/NB_schema_contracts

In [ ]:
from pyspark.sql.functions import col, coalesce, from_json, to_timestamp, row_number, current_timestamp
from pyspark.sql.types import IntegerType, LongType, StringType, StructField, StructType
from pyspark.sql.window import Window
import uuid

dbutils.widgets.text("CATALOG", "workspace")
dbutils.widgets.text("BRONZE_SCHEMA", "bronze")
dbutils.widgets.text("BRONZE_TABLE", "rental")
dbutils.widgets.text("SILVER_SCHEMA", "silver")
dbutils.widgets.text("SILVER_TABLE", "silver_rental")
dbutils.widgets.text("CHECKPOINT_PATH", "/Volumes/workspace/default/mnt/checkpoints/rental_silver")
dbutils.widgets.text("MONITORING_SCHEMA", "monitoring")
dbutils.widgets.text("SCHEMA_POLICY", "additive_only")  # strict, additive_only, permissive
dbutils.widgets.text("ALERT_CHANNEL", "log")  # log, webhook, all
dbutils.widgets.text("WEBHOOK_URL", "")  # Optional Slack/Teams webhook

CONFIG = {
    "catalog": dbutils.widgets.get("CATALOG"),
    "bronze_schema": dbutils.widgets.get("BRONZE_SCHEMA"),
    "bronze_table": dbutils.widgets.get("BRONZE_TABLE"),
    "silver_schema": dbutils.widgets.get("SILVER_SCHEMA"),
    "silver_table": dbutils.widgets.get("SILVER_TABLE"),
    "checkpoint_path": dbutils.widgets.get("CHECKPOINT_PATH"),
    "monitoring_schema": dbutils.widgets.get("MONITORING_SCHEMA"),
    "schema_policy": dbutils.widgets.get("SCHEMA_POLICY"),
    "alert_channel": dbutils.widgets.get("ALERT_CHANNEL"),
    "webhook_url": dbutils.widgets.get("WEBHOOK_URL"),
}

PIPELINE_RUN_ID = str(uuid.uuid4())

bronze_table_fqn = f"{CONFIG['catalog']}.{CONFIG['bronze_schema']}.{CONFIG['bronze_table']}"
silver_schema_fqn = f"{CONFIG['catalog']}.{CONFIG['silver_schema']}"
silver_table_fqn = f"{silver_schema_fqn}.{CONFIG['silver_table']}"
drift_table_fqn = f"{CONFIG['catalog']}.{CONFIG['monitoring_schema']}.schema_drift_log"

In [ ]:
# Initialize monitoring and Silver table
create_schema_drift_table(spark, CONFIG['catalog'], CONFIG['monitoring_schema'])
ensure_schema_exists(CONFIG['catalog'], CONFIG['silver_schema'])

existing_schema = get_existing_schema(spark, silver_table_fqn)
if existing_schema is None:
    create_silver_table_rental(spark, silver_table_fqn)
else:
    print(f"Using existing Silver rental table: {silver_table_fqn}")

In [ ]:
# Schema drift validation
expected_schema = get_schema_contract('silver.rental')
actual_schema = get_existing_schema(spark, silver_table_fqn)

print(f"Validating schema against contract with policy: {CONFIG['schema_policy']}")

try:
    is_valid, drift_result = validate_schema_with_policy(
        spark,
        expected_schema=expected_schema,
        actual_schema=actual_schema,
        table_name=silver_table_fqn,
        drift_table=drift_table_fqn,
        source_system="postgres_cdc",
        pipeline_run_id=PIPELINE_RUN_ID,
        policy=CONFIG['schema_policy'],
        alert_channel=CONFIG['alert_channel'],
        webhook_url=CONFIG['webhook_url'] if CONFIG['webhook_url'] else None
    )
    if drift_result['has_drift']:
        print(f"⚠️ Schema drift detected: {drift_result['severity']}")
        if drift_result['added_columns']:
            print(f"  Added: {drift_result['added_columns']}")
        if drift_result['removed_columns']:
            print(f"  Removed: {drift_result['removed_columns']}")
        if drift_result['type_changes']:
            print(f"  Type changes: {drift_result['type_changes']}")
    else:
        print("✅ Schema validation passed - no drift detected")
except SchemaDriftException as e:
    print(f"🚨 PIPELINE BLOCKED: {e}")
    dbutils.notebook.exit(f"Schema drift blocked pipeline: {e}")

In [ ]:
# Read bronze stream
bronze_stream = (
    spark.readStream
    .option("mergeSchema", "true")
    .table(bronze_table_fqn)
)

# Debezium JSON payload schema for dvdrental.rental
rental_payload_schema = StructType([
    StructField("payload", StructType([
        StructField("before", StructType([
            StructField("rental_id", IntegerType()),
            StructField("rental_date", LongType()),
            StructField("inventory_id", IntegerType()),
            StructField("customer_id", IntegerType()),
            StructField("return_date", LongType()),
            StructField("staff_id", IntegerType()),
            StructField("last_update", LongType())
        ])),
        StructField("after", StructType([
            StructField("rental_id", IntegerType()),
            StructField("rental_date", LongType()),
            StructField("inventory_id", IntegerType()),
            StructField("customer_id", IntegerType()),
            StructField("return_date", LongType()),
            StructField("staff_id", IntegerType()),
            StructField("last_update", LongType())
        ])),
        StructField("op", StringType()),
        StructField("ts_ms", LongType())
    ]))
])

parsed = bronze_stream.select(
    from_json(col("value"), rental_payload_schema).alias("data")
).select("data.payload.*")

In [ ]:
# Parse and transform CDC events
# dvdrental timestamps are microseconds since epoch
events = parsed.select(
    coalesce(col("after.rental_id"), col("before.rental_id")).alias("rental_id"),
    to_timestamp((coalesce(col("after.rental_date"), col("before.rental_date")) / 1000000).cast("double")).alias("rental_date"),
    coalesce(col("after.inventory_id"), col("before.inventory_id")).alias("inventory_id"),
    coalesce(col("after.customer_id"), col("before.customer_id")).alias("customer_id"),
    to_timestamp((coalesce(col("after.return_date"), col("before.return_date")) / 1000000).cast("double")).alias("return_date"),
    coalesce(col("after.staff_id"), col("before.staff_id")).alias("staff_id"),
    to_timestamp((coalesce(col("after.last_update"), col("before.last_update")) / 1000000).cast("double")).alias("last_update"),
    col("op"),
    to_timestamp((col("ts_ms") / 1000).cast("double")).alias("event_time")
)

In [ ]:
def upsert_to_silver(batch_df, batch_id):
    """Upsert CDC events into Silver rental table with schema evolution."""
    if not batch_df.take(1):
        return

    window = Window.partitionBy("rental_id").orderBy(col("event_time").desc())
    latest = (
        batch_df
        .withColumn("rn", row_number().over(window))
        .filter(col("rn") == 1)
        .drop("rn")
    )

    existing_columns = get_existing_columns(spark, silver_table_fqn)
    core_fields = ["rental_id", "rental_date", "inventory_id", "customer_id", "return_date", "staff_id", "last_update"]
    update_set, insert_values = build_merge_clauses(
        existing_columns,
        core_fields,
        include_inserted_dt=True,
        include_updated_dt=True
    )

    delta_table = DeltaTable.forName(spark, silver_table_fqn)
    execute_merge(
        spark, delta_table, latest, update_set, insert_values,
        join_condition="t.rental_id = s.rental_id"
    )

In [ ]:
query = (
    events.writeStream
    .foreachBatch(upsert_to_silver)
    .option("checkpointLocation", CONFIG["checkpoint_path"])
    .option("mergeSchema", "true")
    .trigger(availableNow=True)
    .start()
)

query.awaitTermination()

print(f"✅ Silver rental processing completed. Run ID: {PIPELINE_RUN_ID}")